# Tools 

Can have any arguments but if you want to access state, store, context Need:
- runtime : reserved keyword ToolRuntime to access runtime.state, runtime.store

In [ ]:
from langchain.tools import tool, ToolRuntime

@tools
def example_tool(arg1, args2, runtime:ToolRuntime):

    current_state = runtime.state # Access state
    store = runtime.store # Access store
    user_id = runtime.context.user_id # Access context

    
    is_authenticated = current_state.get("authenticated", False) # get something from state
    existing_prefs = store.get(("preferences",), user_id) # get something from store

    pass  # do something 

# Middleware

## 1. Node style - before_model, after_model, before_agent, after_agent 

Need 
- state: AgentState for accessing state
- runtime : Runtime



In [ ]:
from langchain.agents.middleware import before_model, after_model, AgentState
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any

@after_model
def log_response(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"Model returned: {state['messages'][-1].content}")
    return None

## 2. Wrap style - wrap_model_call, wrap_tool_Call

Need
- request: ModelRequest
- handler: Callable[[ModelRequest], ModelResponse],

if you want to access
- state : request.state
- store : request.runtime.store
- context : request.runtime.context

then if you <b>override something in request</b> like messages, you need to > <b>return handler(request)</b> by handler

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def inject_file_context(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Inject context about files user has uploaded this session."""
    # Read from State: get uploaded files metadata
    uploaded_files = request.state.get("uploaded_files", [])   # K: Access state to get list of uploaded file

    if uploaded_files:
        # Build context about available files
        file_descriptions = []
        for file in uploaded_files:
            file_descriptions.append(
                f"- {file['name']} ({file['type']}): {file['summary']}"
            )

        file_context = f"""Files you have access to in this conversation:
{chr(10).join(file_descriptions)} 

Reference these files when answering questions.""" # K: chr(10) == '\n' newline, so this line is just add string containing all file description 

        # Inject file context before recent messages, K: inject == append back to message and overwrite the request
        messages = [  
            *request.messages,
            {"role": "user", "content": file_context},
        ]
        request = request.override(messages=messages)  # overwrite messages with new message 

    return handler(request)

agent = create_agent(
    model="gpt-4o",
    tools=[...],
    middleware=[inject_file_context]
)